# 01 · From blob to regularized Stokeslet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fairyflies9/fluidlets/blob/main/tutorials/01_blob_to_stokeslet.ipynb)

A **Stokeslet** is the flow produced by a single point force in a very viscous fluid. It is the
building block of the whole method: every structure in Fluidlets (a sphere, a flagellum, a
forest of glycocalyx towers) is represented by a cloud of point forces, and the flow is the sum
of their Stokeslets.

The trouble is that the classical Stokeslet is singular, it blows up like $1/r$ at the force.
Cortez (2001) and Cortez, Fauci & Medovikov (2005) fixed this by spreading each force over a
small smooth **blob** of width $\varepsilon$. In this notebook we

1. write down the 3D blob used in Fluidlets,
2. derive the regularized Stokeslet from it, step by step, and check every step numerically,
3. compare it with the singular Stokeslet using `fluidlets.Stokeslet3D` and `fluidlets.evaluate`,
4. show that away from the force the two differ by $O(\varepsilon^2)$.

**Reference.** R. Cortez, L. Fauci & A. Medovikov, *The method of regularized Stokeslets in three
dimensions: analysis, validation, and application to helical swimming*, Phys. Fluids **17**,
031504 (2005).

In [ ]:
# On Colab or Binder, install Fluidlets first (skipped when it is already installed).
import importlib.util, subprocess, sys
if importlib.util.find_spec("fluidlets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/fairyflies9/fluidlets"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.integrate import quad

import fluidlets as fl
from fluidlets.kernels.stokeslet3d import H1, H2   # the functions the solver actually uses

## 1. The Stokes equations with a smoothed force

At low Reynolds number the fluid velocity $\mathbf u$ and pressure $p$ satisfy the Stokes
equations. Instead of a point force $\mathbf f\,\delta(\mathbf x - \mathbf x_0)$ we push the fluid
with a force $\mathbf f$ spread over a blob $\phi_\varepsilon$ centred at $\mathbf x_0$:

$$
\mu \Delta \mathbf u = \nabla p - \mathbf f\, \phi_\varepsilon(\mathbf x - \mathbf x_0),
\qquad \nabla\cdot\mathbf u = 0 .
$$

The blob is radial, has total mass one, and tends to the Dirac delta as $\varepsilon\to 0$.
Fluidlets uses the blob of Cortez, Fauci & Medovikov (2005, Eq. 9):

$$
\phi_\varepsilon(r) = \frac{15\,\varepsilon^4}{8\pi\,(r^2+\varepsilon^2)^{7/2}} .
$$

Let us look at it and check that its integral over all of space, $\int_0^\infty 4\pi r^2 \phi_\varepsilon\,dr$, is one.

In [ ]:
def phi(r, eps):
    """The 3D blob of Cortez, Fauci & Medovikov (2005)."""
    return 15 * eps**4 / (8 * np.pi * (r**2 + eps**2) ** 3.5)

r = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
for eps in [0.4, 0.2, 0.1]:
    mass = quad(lambda s: 4 * np.pi * s**2 * phi(s, eps), 0, np.inf)[0]
    ax.plot(r, phi(r, eps), label=f"eps = {eps}  (mass = {mass:.6f})")
ax.set_yscale("log"); ax.set_xlabel("r"); ax.set_ylabel(r"$\phi_\varepsilon(r)$")
ax.legend(); ax.set_title("Narrower blobs are taller: the mass stays 1")
plt.show()

## 2. Deriving the regularized Stokeslet

The derivation needs two auxiliary radial functions, defined by

$$
\Delta G_\varepsilon = \phi_\varepsilon, \qquad \Delta B_\varepsilon = G_\varepsilon .
$$

(For a point force, $G$ is the Green's function of the Laplacian, $-1/(4\pi r)$, and $B$ is
the biharmonic Green's function, $-r/(8\pi)$.)

**Pressure.** Take the divergence of the momentum equation. Since $\nabla\cdot\mathbf u = 0$,
the left side vanishes and $\Delta p = \mathbf f\cdot\nabla\phi_\varepsilon$, so
$p = \mathbf f\cdot\nabla G_\varepsilon$.

**Velocity.** Put the pressure back: $\mu\Delta\mathbf u = (\mathbf f\cdot\nabla)\nabla G_\varepsilon - \mathbf f\,\phi_\varepsilon$.
Both terms on the right are a Laplacian of something we know, so

$$
\mu\,\mathbf u = (\mathbf f\cdot\nabla)\nabla B_\varepsilon - \mathbf f\, G_\varepsilon .
$$

**Radial functions.** For a radial $g(r)$ the Laplacian is $\Delta g = r^{-2}(r^2 g')'$, and the Hessian is

$$
\nabla\nabla B = \frac{B'}{r}\,\mathbf I + \Big(B'' - \frac{B'}{r}\Big)\frac{\mathbf d\,\mathbf d^{T}}{r^2},
\qquad \mathbf d = \mathbf x - \mathbf x_0,\; r = |\mathbf d| .
$$

So the velocity has the form used throughout Fluidlets,

$$
\mathbf u = H_1(r)\,\mathbf f + H_2(r)\,(\mathbf f\cdot\mathbf d)\,\mathbf d, \qquad
\mu H_1 = \frac{B'}{r} - G, \qquad \mu H_2 = \frac{1}{r^2}\Big(B'' - \frac{B'}{r}\Big).
$$

### Solving for $G_\varepsilon$ and $B_\varepsilon$

Write $R = \sqrt{r^2+\varepsilon^2}$. Integrating $(r^2 G')' = r^2\phi_\varepsilon$ once from 0 gives

$$
r^2 G_\varepsilon' = \frac{r^3(2r^2 + 5\varepsilon^2)}{8\pi R^5},
$$

and integrating again with $G_\varepsilon\to 0$ at infinity,

$$
G_\varepsilon(r) = -\frac{2r^2 + 3\varepsilon^2}{8\pi R^3}.
$$

For $B_\varepsilon$, notice that $R' = r/R$ and $R'' = \varepsilon^2/R^3$, so
$\Delta R = R'' + 2R'/r = (2r^2 + 3\varepsilon^2)/R^3$. That is exactly $-8\pi G_\varepsilon$, hence

$$
B_\varepsilon(r) = -\frac{R}{8\pi} = -\frac{\sqrt{r^2+\varepsilon^2}}{8\pi}.
$$

As $\varepsilon\to0$ these become $-1/(4\pi r)$ and $-r/(8\pi)$, the point-force answers. Rather
than trusting the algebra, let us check each step with numbers.

In [ ]:
eps = 0.3
G = lambda r: -(2 * r**2 + 3 * eps**2) / (8 * np.pi * (r**2 + eps**2) ** 1.5)
B = lambda r: -np.sqrt(r**2 + eps**2) / (8 * np.pi)

def radial_laplacian(g, r, h=1e-4):
    """g'' + 2 g'/r by centred differences."""
    gp = (g(r + h) - g(r - h)) / (2 * h)
    gpp = (g(r + h) - 2 * g(r) + g(r - h)) / h**2
    return gpp + 2 * gp / r

rs = np.array([0.1, 0.5, 1.0, 3.0])
print("   r     Lap G       phi      |   Lap B        G")
for rr in rs:
    print(f"{rr:5.1f}  {radial_laplacian(G, rr):9.3e} {phi(rr, eps):9.3e}  | "
          f"{radial_laplacian(B, rr):9.5f} {G(rr):9.5f}")

The two columns agree in each pair, so $\Delta G_\varepsilon = \phi_\varepsilon$ and $\Delta B_\varepsilon = G_\varepsilon$.

### The result: $H_1$ and $H_2$

Now differentiate $B_\varepsilon = -R/(8\pi)$: $B' = -r/(8\pi R)$ and $B'' = -\varepsilon^2/(8\pi R^3)$. Substituting,

$$
\boxed{\;H_1(r) = \frac{r^2 + 2\varepsilon^2}{8\pi\mu\,(r^2+\varepsilon^2)^{3/2}}, \qquad
H_2(r) = \frac{1}{8\pi\mu\,(r^2+\varepsilon^2)^{3/2}}\;}
$$

These are the functions `H1(r2, eps2)` and `H2(r2, eps2)` in `fluidlets/kernels/stokeslet3d.py`
(written for $\mu = 1$ and taking $r^2$, $\varepsilon^2$ to avoid square roots). Check them against
the formulas $B'/r - G$ and $(B'' - B'/r)/r^2$, using finite differences of $B$:

In [ ]:
h = 1e-4
Bp = (B(rs + h) - B(rs - h)) / (2 * h)
Bpp = (B(rs + h) - 2 * B(rs) + B(rs - h)) / h**2
print("H1 from B:", Bp / rs - G(rs))
print("H1 code:  ", H1(rs**2, eps**2))
print("H2 from B:", (Bpp - Bp / rs) / rs**2)
print("H2 code:  ", H2(rs**2, eps**2))

## 3. Regularized vs singular Stokeslet

Setting $\varepsilon = 0$ gives the classical (singular) Stokeslet, also called the Oseen tensor:

$$
H_1^0 = \frac{1}{8\pi\mu r}, \qquad H_2^0 = \frac{1}{8\pi\mu r^3}.
$$

The regularized version is finite at the force: $H_1(0) = 1/(4\pi\mu\varepsilon)$. Far away the two
are indistinguishable.

In [ ]:
r = np.logspace(-3, 1, 300)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8))
for eps in [0.1, 0.01]:
    axs[0].loglog(r, H1(r**2, eps**2), label=f"eps = {eps}")
    axs[1].loglog(r, H2(r**2, eps**2), label=f"eps = {eps}")
axs[0].loglog(r, 1 / (8 * np.pi * r), "k--", label="singular")
axs[1].loglog(r, 1 / (8 * np.pi * r**3), "k--", label="singular")
axs[0].set_title("$H_1(r)$"); axs[1].set_title("$H_2(r)$")
for ax in axs:
    ax.set_xlabel("r"); ax.legend()
plt.tight_layout(); plt.show()

### Velocity from `fluidlets.evaluate`

In practice you never call `H1` and `H2` yourself. A kernel object (`fl.Stokeslet3D()`) knows how
to build the $3\times3$ block $H_1\mathbf I + H_2\,\mathbf d\mathbf d^T$, and `fl.evaluate(kernel,
targets, sources, forces, eps, mu)` sums the contributions of all sources at all targets in
compiled (Numba) code. Arrays are always shaped `(N, 3)`: one row per point.

Put a unit force $\mathbf f = \mathbf e_z$ at the origin and look at $u_z$ along two lines:
perpendicular to the force (the $x$-axis, where $\mathbf f\cdot\mathbf d = 0$ so $u_z = H_1$) and
along the force (the $z$-axis, where $u_z = H_1 + H_2 r^2$, twice as big far away).

In [ ]:
kernel = fl.Stokeslet3D()
mu = 1.0
source = np.zeros((1, 3))
force = np.array([[0.0, 0.0, 1.0]])

def singular_stokeslet(targets, f, mu=1.0):
    """Velocity of a point force f at the origin (the Oseen tensor)."""
    r = np.linalg.norm(targets, axis=1)[:, None]
    return (f / r + (targets @ f)[:, None] * targets / r**3) / (8 * np.pi * mu)

s = np.linspace(-1, 1, 400)                     # even count: skips s = 0, where the singular one is infinite
on_x = np.c_[s, 0 * s, 0 * s]
on_z = np.c_[0 * s, 0 * s, s]

fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, line, label in [(axs[0], on_x, "x"), (axs[1], on_z, "z")]:
    for eps in [0.2, 0.05]:
        u = fl.evaluate(kernel, line, source, force, eps, mu)
        ax.plot(s, u[:, 2], label=f"eps = {eps}")
    ax.plot(s, singular_stokeslet(line, force[0])[:, 2], "k--", label="singular")
    ax.set_ylim(0, 2); ax.set_xlabel(label); ax.set_title(f"$u_z$ along the {label}-axis")
axs[0].legend(); plt.tight_layout(); plt.show()

Every kernel also has a plain NumPy `velocity` method (one target, one source). It is kept as a
*reference implementation*: it is easy to read, and the tests use it to check the fast Numba path.

In [ ]:
x = np.array([0.3, -0.2, 0.5]); f = np.array([1.0, 2.0, -0.5]); eps = 0.1
print("reference:", kernel.velocity(x, np.zeros(3), f, eps, mu))
print("evaluate: ", fl.evaluate(kernel, x[None, :], source, f[None, :], eps, mu)[0])

### The flow around one regularized Stokeslet

Here is the velocity in the plane $y = 0$ for a force pointing up. In unbounded fluid there is
no return flow: everything moves upward, fastest along the line of the force, and the speed decays
only like $1/r$ (note the log colour scale). That slow decay is why distant parts of a structure
in Stokes flow interact so strongly, and why walls (tutorial 03) matter so much.

In [ ]:
g = np.linspace(-2, 2, 41)
Xg, Zg = np.meshgrid(g, g)
grid = np.c_[Xg.ravel(), 0 * Xg.ravel(), Zg.ravel()]
u = fl.evaluate(kernel, grid, source, force, 0.1, mu)
ux, uz = u[:, 0].reshape(Xg.shape), u[:, 2].reshape(Xg.shape)

fig, ax = plt.subplots(figsize=(5, 5))
pc = ax.pcolormesh(Xg, Zg, np.hypot(ux, uz), shading="auto", cmap="viridis",
                   norm=LogNorm(vmin=0.02, vmax=0.5))
ax.streamplot(g, g, ux, uz, color="w", density=1.2, linewidth=0.7)
ax.arrow(0, -0.25, 0, 0.4, width=0.03, color="r")
ax.set_aspect("equal"); ax.set_xlabel("x"); ax.set_ylabel("z")
fig.colorbar(pc, ax=ax, label="|u|"); ax.set_title("Regularized Stokeslet, eps = 0.1")
plt.show()

## 4. How fast does the regularized Stokeslet converge?

Fix a distance $r$ and let $\varepsilon\to0$. With $x = \varepsilon^2/r^2$,

$$
\frac{H_1}{H_1^0} = \frac{1 + 2x}{(1+x)^{3/2}} = 1 + \tfrac12 x + O(x^2),
\qquad
\frac{H_2}{H_2^0} = \frac{1}{(1+x)^{3/2}} = 1 - \tfrac32 x + O(x^2).
$$

So the relative error is $O(\varepsilon^2/r^2)$: **second order in $\varepsilon$ away from the
force**, but $O(1)$ within a few $\varepsilon$ of it. Let us measure it on random points with
$0.5 \le r \le 2$, for a random force.

In [ ]:
rng = np.random.default_rng(0)
dirs = rng.normal(size=(500, 3)); dirs /= np.linalg.norm(dirs, axis=1)[:, None]
targets = dirs * rng.uniform(0.5, 2.0, size=(500, 1))
f = rng.normal(size=(1, 3))
exact = singular_stokeslet(targets, f[0])

epsilons = np.logspace(-3, -0.5, 8)
errors = []
for eps in epsilons:
    u = fl.evaluate(kernel, targets, source, f, eps, mu)
    errors.append(np.abs(u - exact).max() / np.abs(exact).max())
errors = np.array(errors)

slope = np.polyfit(np.log(epsilons[:5]), np.log(errors[:5]), 1)[0]
print(f"observed order of convergence: {slope:.3f}")
assert abs(slope - 2) < 0.05

fig, ax = plt.subplots(figsize=(5, 3.8))
ax.loglog(epsilons, errors, "o-", label="max relative error, 0.5 < r < 2")
ax.loglog(epsilons, errors[0] * (epsilons / epsilons[0]) ** 2, "k--", label=r"$\propto\varepsilon^2$")
ax.set_xlabel(r"$\varepsilon$"); ax.legend(); plt.show()

The slope is 2: halve $\varepsilon$ and the far-field error drops by four.

This is a statement about **one** Stokeslet. When many Stokeslets represent a surface, a second
error appears, from replacing a surface integral by a sum over discrete points, and it competes
with this one. That trade-off, and how to choose $\varepsilon$, is the subject of
[tutorial 02](02_choosing_eps.ipynb).

## Summary

* The blob $\phi_\varepsilon = 15\varepsilon^4 / (8\pi(r^2+\varepsilon^2)^{7/2})$ gives
  $G_\varepsilon = -(2r^2+3\varepsilon^2)/(8\pi R^3)$ and $B_\varepsilon = -R/(8\pi)$, $R = \sqrt{r^2+\varepsilon^2}$.
* The regularized Stokeslet is $\mathbf u = H_1\mathbf f + H_2(\mathbf f\cdot\mathbf d)\mathbf d$ with
  $H_1 = (r^2+2\varepsilon^2)/(8\pi\mu R^3)$, $H_2 = 1/(8\pi\mu R^3)$.
* It is finite at the force and differs from the singular Stokeslet by $O(\varepsilon^2/r^2)$.
* In code: `fl.evaluate(fl.Stokeslet3D(), targets, sources, forces, eps, mu)`.

## Exercises

1. **Mass and width.** Show by hand that $\int_{\mathbb R^3}\phi_\varepsilon\,dV = 1$ (substitute
   $r = \varepsilon\tan\theta$). What fraction of the mass lies within $r < \varepsilon$? Within $r < 3\varepsilon$?
2. **Incompressibility.** Use `fl.evaluate` and centred differences to check numerically that
   $\nabla\cdot\mathbf u = 0$ for the regularized Stokeslet, including at points closer than $\varepsilon$ to the force.
3. **Pressure.** From $p = \mathbf f\cdot\nabla G_\varepsilon$, find the regularized pressure in closed
   form. Plot it along the $z$-axis next to the singular pressure $\mathbf f\cdot\mathbf d/(4\pi r^3)$.
4. **A different blob.** Repeat the derivation for $\phi_\varepsilon = 3\varepsilon^2/(4\pi(r^2+\varepsilon^2)^{5/2})$.
   (Hint: $\Delta (r^2+\varepsilon^2)^{-1/2} = -3\varepsilon^2 (r^2+\varepsilon^2)^{-5/2}$.) Is the far-field
   error still $O(\varepsilon^2)$? Why might the answer depend on how fast the blob decays?
5. **Two forces.** Place $+\mathbf e_z$ at $(0,0,0.5)$ and $-\mathbf e_z$ at $(0,0,-0.5)$ (a force
   dipole, like a swimming bacterium pushing fluid off both ends). Plot the flow as in Section 3.
   How fast does the speed decay with distance now?